In [ ]:
from pathlib import Path
import csv
import sys
sys.dont_write_bytecode = True

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "data").is_dir()),
    None,
)

CSV_DIR = PROJECT_ROOT / "data"
FIG_DIR = PROJECT_ROOT / "resultados"

summary_path = CSV_DIR / "resumen_simulacion.csv"
with summary_path.open("r", newline="", encoding="utf-8") as file:
    summary_rows = csv.DictReader(file)
    parameters = {row["metrica"]: row["valor"] for row in summary_rows}

path = CSV_DIR / "funcion_distribucion_radial.csv"
r_values, g_values, pair_counts = [], [], []

with path.open("r", newline="", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    required = {"r", "g(r)", "pares_promedio"}
    
    for row in reader:
        r_values.append(float(row["r"]))
        g_values.append(float(row["g(r)"]))
        pair_counts.append(float(row["pares_promedio"]))

rdf = np.column_stack((r_values, g_values, pair_counts))

order = np.argsort(rdf[:, 0])
r_values = rdf[order, 0]
g_values = rdf[order, 1]
pair_counts = rdf[order, 2]
print(f"N={int(parameters['N'])}; l={parameters['l']:g}; "
      f"ancho bin={parameters['ancho_bin_gr']:g}; método={parameters['metodo']}; "
      f"{len(r_values)} bins; alcance radial: 0 a {r_values[-1]:g}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].plot(r_values, g_values, marker="o", markersize=3, label="$g(r)$")
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1,
                label="Gas ideal ($g=1$)")
axes[0].set(title=f"Función de distribución radial 2D — N={int(parameters['N'])}, "
            f"l={parameters['l']:g}, dr={parameters['ancho_bin_gr']:g}",
            xlabel="Distancia $r$", ylabel="$g(r)$")
axes[0].grid(alpha=0.25)
axes[0].legend()

if len(r_values) > 1:
    bin_width = np.diff(r_values, append=2 * r_values[-1] - r_values[-2])
else:
    bin_width = np.array([1.0])
axes[1].bar(r_values, pair_counts, width=bin_width * 0.9,
            color="#3973ac", edgecolor="white")
axes[1].set(title="Pares ordenados promedio por bin",
            xlabel="Distancia $r$", ylabel="Pares promedio")
axes[1].grid(axis="y", alpha=0.25)
plt.tight_layout()
fig.savefig(FIG_DIR / "distribucion_radial.png", dpi=160, bbox_inches="tight")
plt.show()